In [38]:
!pip install scikit-learn==1.6.1 xgboost shap joblib

Defaulting to user installation because normal site-packages is not writeable


In [39]:
import pandas as pd
import numpy as np
import joblib
import shap
import warnings
warnings.filterwarnings('ignore')


from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix
from xgboost import XGBClassifier

In [40]:
# Load Data
df = pd.read_csv('D:\Bank Loan Approval Intelligence System\Data\loan_data_cleaned.csv')
x = df.drop('loan_status', axis=1)
y = df['loan_status']

# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42, stratify=y)

print(f'Training set size: {X_train.shape[0]} samples | Test: {X_test.shape[0]} samples')

Training set size: 35628 samples | Test: 8907 samples


In [41]:
numeric_features = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X_train.select_dtypes(include=['object']).columns.tolist()

In [42]:
preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), numeric_features),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features),
])

In [61]:
X_transformed = preprocessor.fit_transform(X_train)

In [43]:
# ── 3 MODELS ──────────────────────────────────────────────────
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, class_weight='balanced'),
    'Random Forest':       RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42, n_jobs=-1),
    'XGBoost':             XGBClassifier(n_estimators=100, scale_pos_weight=3, random_state=42, eval_metric='logloss', verbosity=0),
}

In [44]:
# ── TRAIN & COMPARE ───────────────────────────────────────────
results = {}
for name, model in models.items():
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', model)
    ])
    pipeline.fit(X_train, y_train)
    y_prob = pipeline.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, y_prob)
    results[name] = {'pipeline': pipeline, 'auc': auc}
    print(f"{name:25s} --> ROC-AUC: {auc:.4f}")

Logistic Regression       --> ROC-AUC: 0.9553
Random Forest             --> ROC-AUC: 0.9750
XGBoost                   --> ROC-AUC: 0.9789


In [45]:
best_name = max(results, key=lambda x: results[x]['auc'])
best_pipeline = results[best_name]['pipeline']
print(f"\n🏆 Best model: {best_name} (AUC: {results[best_name]['auc']:.4f})")


🏆 Best model: XGBoost (AUC: 0.9789)


In [46]:
xgb_tuned = Pipeline(
    steps =['preprocessor', preprocessor,
    ('classifier', XGBClassifier(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=3,
        random_state=42, 
        eval_metric='logloss', 
        verbosity=0)
    )
])

In [47]:
xgb_tuned = Pipeline(steps=[
	('preprocessor', preprocessor),
	('classifier', XGBClassifier(
		n_estimators=200,
		max_depth=6,
		learning_rate=0.05,
		subsample=0.8,
		colsample_bytree=0.8,
		scale_pos_weight=3,
		random_state=42,
		eval_metric='logloss',
		verbosity=0
	))
])

xgb_tuned.fit(X_train, y_train)
y_prob_tuned = xgb_tuned.predict_proba(X_test)[:, 1]
y_pred_tuned = (y_prob_tuned >= 0.4).astype(int)

In [48]:
auc_tuned = roc_auc_score(y_test, y_prob_tuned)
print(f"\nXGBoost Tuned --> ROC-AUC: {auc_tuned:.4f}")
print("\nClassification Report: (Threshold=0.4)")
print(classification_report(y_test, y_pred_tuned, target_names = ['Rejected', 'Approve']))


XGBoost Tuned --> ROC-AUC: 0.9771

Classification Report: (Threshold=0.4)
              precision    recall  f1-score   support

    Rejected       0.98      0.87      0.92      6919
     Approve       0.68      0.94      0.79      1988

    accuracy                           0.89      8907
   macro avg       0.83      0.90      0.85      8907
weighted avg       0.91      0.89      0.89      8907



In [49]:
# Get feature names after preprocessing


feature_names = (
    numeric_features + 
    list(xgb_tuned.named_steps['preprocessor']
    .named_transformers_['cat']
    .get_feature_names_out(categorical_features)))

In [50]:
# Transform test data
X_test_tansformed = xgb_tuned.named_steps['preprocessor'].transform(X_test)

In [51]:
# SHAP explainer
explainer = shap.TreeExplainer(xgb_tuned.named_steps['classifier'])
shap_values = explainer.shap_values(X_test_tansformed)

print("\nSHAP values computed for test set. Ready for visualization.")
print(f'   SHAP values shape: {shap_values.shape}')


SHAP values computed for test set. Ready for visualization.
   SHAP values shape: (8907, 27)


In [52]:
# ── SHAP GLOBAL FEATURE IMPORTANCE ───────────────────────────
shap_importance = pd.DataFrame({
    'feature': feature_names,
    'importance': np.abs(shap_values).mean(axis=0)
}).sort_values('importance', ascending=False)

print(f"\nTop 10 most important features (SHAP):")
print(shap_importance.head(10).to_string(index=False))




Top 10 most important features (SHAP):
                       feature  importance
previous_loan_defaults_on_file    3.998731
                 loan_int_rate    0.632928
                 person_income    0.466852
                loan_to_income    0.455428
    person_home_ownership_RENT    0.261925
                  credit_score    0.191022
           loan_intent_VENTURE    0.181485
     person_home_ownership_OWN    0.175420
           loan_percent_income    0.131354
   loan_intent_HOMEIMPROVEMENT    0.095233


In [53]:
# ── SAVE MODEL ────────────────────────────────────────────────
joblib.dump(xgb_tuned, 'loan_approval_model.pkl', compress=5)
print(f"\n Model saved → loan_approval_model.pkl")



 Model saved → loan_approval_model.pkl


In [54]:
# Save SHAP explainer and feature names for the app

joblib.dump(explainer, 'shap_explainer.pkl', compress=5)
joblib.dump(feature_names, 'feature_names.pkl')

print("Saved --> shap_explainer.pkl")
print("Saved --> feature_names.pkl")
print(f"\nTotal files ready for app:")
print("  - loan_approval_model.pkl")
print("  - shap_explainer.pkl") 
print("  - feature_names.pkl")
print("  - data/sql_q1_education.csv")
print("  - data/sql_q2_loan_purpose.csv")
print("  - data/sql_q3_home_ownership.csv")
print("  - data/sql_q4_income_band.csv")
print("  - data/sql_q5_defaults.csv")

Saved --> shap_explainer.pkl
Saved --> feature_names.pkl

Total files ready for app:
  - loan_approval_model.pkl
  - shap_explainer.pkl
  - feature_names.pkl
  - data/sql_q1_education.csv
  - data/sql_q2_loan_purpose.csv
  - data/sql_q3_home_ownership.csv
  - data/sql_q4_income_band.csv
  - data/sql_q5_defaults.csv
